### TP INF 372 : Analyse de la Consommation Data

| Nom Complet | Matricule |
| :--- | :--- |
| KAMDOM NGUETCHESSI MERVEILLE PRISCA | 23V2060 |
| YOKO AYMERICK JEFFREY | 23U2983 |
| KENNE YEMTSA LOWEL RAYANN | 23U2477 |

---

# Feature Engineering et Préparation(SVM et Regression Logistique)

Ce notebook transforme les données brutes issues de la phase de nettoyage en deux jeux de données exploitables pour la classification. L'étape clé est le **mapping temporel** (division de la journée en 4 périodes) et la création de variables métier comme le **débit (throughput)**. Nous créons deux versions du dataset pour comparer l'impact réel de l'ingénierie des caractéristiques sur la performance finale des modèles.

### Objectifs :
1. Transformer les types de données (Heures -> Périodes).
2. Gérer les encodages résiduels pour éviter la perte de données.
3. Créer des datasets distincts pour valider l'intuition métier.

### Questions de Recherche :
- Comment transformer l'heure brute en une catégorie comportementale pertinente ?
- Est-ce que des variables dérivées (Log, Débit) apportent plus de signal que les volumes bruts ?

---

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/processed/composed_dataset.csv")

## 1. Encodage et Target Mapping

### Intuition sur le nettoyage des chaines de caractères
Lors de la composition, nous avons vu que certains fichiers utilisaient des encodages différents (ex: `ConnectÃ©` au lieu de `Connecté`). Nous corrigeons cela ici pour ne pas perdre ces lignes lors de la conversion en nombres.

In [2]:
def extract_hour(time_str):
    return int(time_str.split(":")[0])

def map_hour_to_period(hour):
    if 0 <= hour < 6: return "Nuit"
    if 6 <= hour < 12: return "Matin"
    if 12 <= hour < 18: return "Après-midi"
    return "Soir"

df['_heure_int'] = df['heure'].apply(extract_hour)
df['periode_journee'] = df['_heure_int'].apply(map_hour_to_period)

CONNEXION_MAP = {"Connecté": 1, "ConnectÃ©": 1, "Déconnecté": 0, "DÃ©connectÃ©": 0}
df['statut_connexion'] = df['statut_connexion'].map(CONNEXION_MAP)
df['periode_semaine'] = df['jour_semaine'].isin(['Samedi', 'Dimanche']).astype(int)

df = df.dropna(subset=['statut_connexion'])

### Pourquoi des tranches de 6 heures ?

Pour valider scientifiquement ce découpage, nous vérifions si la consommation moyenne diffère réellement entre ces périodes. Un bon découpage doit maximiser la différence entre les groupes (variance inter-classe).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Analyse de la consommation moyenne par tranche de 6h
stats = df.groupby('periode_journee')['download_Mo'].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=stats, x='periode_journee', y='download_Mo', hue='periode_journee', palette='viridis', legend=False)
plt.title("Consommation moyenne par période (Tranches de 6h)")
plt.ylabel("Download Moyen (Mo)")
plt.show()

print("Consommation Moyenne par Période :")
print(df.groupby('periode_journee')['download_Mo'].mean())

**Observation** : On remarque que la consommation n'est pas uniforme. La "Nuit" et le "Soir" présentent souvent des pics liés aux loisirs/mises à jour, tandis que le "Matin" est plus calme. Ce découpage capture donc bien des **comportements distincts**.

## 2. Construction de la Version "Basic"

### Justification du Groupe de Contrôle
Pourquoi garder un dataset avec des caractéristiques brutes ? En science, il est crucial d'avoir un **groupe de contrôle**. 

**Notre Jugement** : Si un modèle complexe (comme le SVM) ne fait pas mieux sur le dataset "Enhanced" que sur le dataset "Basic", c'est que nos transformations sont inutiles ou mal choisies. Le dataset Basic sert de **référence de performance minimale**.

In [ ]:
basic_cols = ['statut_connexion', 'download_Mo', 'upload_Mo', 'periode_semaine', 'periode_journee']
df[basic_cols].to_csv("../../data/processed/dataset_classification_basic.csv", index=False)

## 3. L'Ingénierie des caractéristiques ("Enhanced")

### Pourquoi ces transformations ?
1.  **Débit Moyen (Throughput)** : Une session de 100Mo sur 1 min est différente d'une session de 100Mo sur 60 min. Le débit reflète mieux l'activité (Vidéoconférence vs Mises à jour en arrière-plan).
2.  **Log-Scaling** : Les modèles comme le SVM détestent les échelles trop larges. Le Log permet de rapprocher les petites et les grandes valeurs pour que le modèle puisse trouver une frontière plus facilement.
3.  **Durée (Duree_minutes)** : On l'ajoute car le temps passé en ligne est fortement corrélé aux périodes de loisir (Soir/Nuit).

In [ ]:
df['throughput'] = (df['download_Mo'] + df['upload_Mo']) / (df['duree_minutes'] + 0.1)
df['log_down'] = np.log1p(df['download_Mo'])
df['log_up'] = np.log1p(df['upload_Mo'])

enh_cols = ['statut_connexion', 'duree_minutes', 'log_down', 'log_up', 'throughput', 'periode_semaine', 'periode_journee']
df[enh_cols].to_csv("../../data/processed/dataset_classification_enhanced.csv", index=False)

## Conclusion du Notebook 03
Nous avons armé nos modèles avec deux visions différentes de la même réalité. 

**Résultat attendu** : Une amélioration nette de la précision (Accuracy) grâce au dataset "Enhanced", validant ainsi notre expertise métier sur la donnée.

--- 

### Difficultés Rencontrées
1. **Encodages Fantômes** : Malgré le nettoyage initial, des variations comme `ConnectÃ©` persistaient. Nous avons dû utiliser un mapping robuste pour ne pas biaiser la variable cible.
2. **Division Temporelle** : Choisir entre un découpage fixe (ex: toutes les 6h) et un découpage métier (ex: heures pleines/creuses). Nous avons retenu le découpage fixe pour sa neutralité initiale.

### Évaluation de cette phase
Les fichiers générés sont maintenant compacts et prêts pour l'entraînement. La prochaine étape (Notebook 04) nous dira si ces efforts de préparation portent leurs fruits.